# Jupyter Chatbooks Overview

Anton Antonov  
March 2026

---

## Abstract


This presentation introduces notebook-wide LLM-persona chat workflows in [Jupyter Chatbooks](https://github.com/antononcube/Python-JupyterChatbook), explains the execution flow, and discusses key design and implementation decisions.

---

## Who am I?

- MSc Mathematics (General Algebra)

- MSc Computer Science (Databases)

- PhD Applied Mathematics (Large-Scale Air Pollution Models)

- Former Kernel Developer of Mathematica 

    - Now Wolfram Language

- Over 30 years of numerical and applied mathematics in industrial settings

---

## Plan of the presentation:

1. Notebook-wide LLM-persona chat using magic cells -- interactive demo
2. How it works?
3. Design and implementation discussion

---

## Notebook-wide LLM-persona chat

- LLM access via Jupyter magic cells

- Easy to create LLM personas
    
    - Identifiers, models, parameters, prompts

- Global-notebook context

- Chat-cells DSL 

- Management of chat objects

*Follow ["JupyterChatbook-simple-demo.ipynb"](https://github.com/antononcube/PythonForPrediction-blog/blob/main/Presentations/JupyterChatbook-simple-demo.ipynb).*

---

## How it works?

### The main workflow

```mermaid
flowchart LR
    OpenAI{{OpenAI}}
    Gemini{{Gemini}}
    Ollama{{Ollama}}
    LLMFunc[[LLMFunctions]]
    LLMProm[[LLMPrompts]]
    CODB[(Chat objects)]
    PDB[(Prompts)]
    CCell[/Chat cell/]
    CRCell[/Chat result cell/]
    CIDQ{Chat ID<br/>specified?}
    CIDEQ{Chat ID<br/>exists in DB?}
    RECO[Retrieve existing<br/>chat object]
    COEval[Message<br/>evaluation]
    PromParse[Prompt<br/>DSL spec parsing]
    KPFQ{Known<br/>prompts<br/>found?}
    PromExp[Prompt<br/>expansion]
    CNCO[Create new<br/>chat object]
    CIDNone["Assume chat ID<br/>is 'NONE'"] 
    subgraph Chatbook frontend    
        CCell
        CRCell
    end
    subgraph Chatbook backend
        CIDQ
        CIDEQ
        CIDNone
        RECO
        CNCO
        CODB
    end
    subgraph Prompt processing
        PDB
        LLMProm
        PromParse
        KPFQ
        PromExp 
    end
    subgraph LLM interaction
      COEval
      LLMFunc
      OpenAI
      Gemini
      Ollama
    end
    CCell --> CIDQ
    CIDQ --> |yes| CIDEQ
    CIDEQ --> |yes| RECO
    RECO --> PromParse
    COEval --> CRCell
    CIDEQ -.- CODB
    CIDEQ --> |no| CNCO
    LLMFunc -.- CNCO -.- CODB
    CNCO --> PromParse --> KPFQ
    KPFQ --> |yes| PromExp
    KPFQ --> |no| COEval
    PromParse -.- LLMProm 
    PromExp -.- LLMProm
    PromExp --> COEval 
    LLMProm -.- PDB
    CIDQ --> |no| CIDNone
    CIDNone --> CIDEQ
    COEval -.- LLMFunc
    LLMFunc <-.-> OpenAI
    LLMFunc <-.-> Gemini
    LLMFunc <-.-> Ollama
```

### Meta-cells

```mermaid
flowchart LR
    LLMFunc[[LLMFunctionObjects]]
    CODB[(Chat objects)]
    CCell[/Chat meta cell/]
    CRCell[/Chat meta cell result/]
    CIDQ{Chat ID<br/>specified?}
    KCOMQ{Known<br/>chat object<br/>method?}
    AKWQ{Option '--all'<br/>specified?} 
    KCODBMQ{Known<br/>chat objects<br/>DB method?}
    CIDEQ{Chat ID<br/>exists in DB?}
    RECO[Retrieve existing<br/>chat object]
    COEval[Chat object<br/>method<br/>invocation]
    CODBEval[Chat objects DB<br/>method<br/>invocation]
    CNCO[Create new<br/>chat object]
    CIDNone["Assume chat ID<br/>is 'NONE'"] 
    NoCOM[/Cannot find<br/>chat object<br/>message/]
    CntCmd[/Cannot interpret<br/>command<br/>message/]
    subgraph Chatbook
        CCell
        NoCOM
        CntCmd
        CRCell
    end
    CCell --> CIDQ
    CIDQ --> |yes| CIDEQ  
    CIDEQ --> |yes| RECO
    RECO --> KCOMQ
    KCOMQ --> |yes| COEval --> CRCell
    KCOMQ --> |no| CntCmd
    CIDEQ -.- CODB
    CIDEQ --> |no| NoCOM
    LLMFunc -.- CNCO -.- CODB
    CNCO --> COEval
    CIDQ --> |no| AKWQ
    AKWQ --> |yes| KCODBMQ
    KCODBMQ --> |yes| CODBEval
    KCODBMQ --> |no| CntCmd
    CODBEval -.- CODB
    CODBEval --> CRCell
    AKWQ --> |no| CIDNone
    CIDNone --> CIDEQ
    COEval -.- LLMFunc
```

---

## Design and implementation discussion

- Original designs by Wolfram Research, Inc.
- Translation to Raku:
    - [LLM-functions framework](https://raku.land/zef:antononcube/LLM::Functions)
    - [LLM prompts system](https://raku.land/zef:antononcube/LLM::Prompts)
    - [Jupyter Chatbook](https://raku.land/zef:antononcube/Jupyter::Chatbook)
- Translation to Python:
    - [LLM-functions framework](https://pypi.org/project/LLMFunctionObjects/)
    - [LLM prompts system](https://pypi.org/project/LLMPrompts/)
    - [Jupyter Chatbook](https://pypi.org/project/JupyterChatbook/)

----

## References

### Articles

[SW1] Stephen Wolfram, ["The New World of LLM Functions: Integrating LLM Technology into the Wolfram Language"](https://writings.stephenwolfram.com/2023/05/the-new-world-of-llm-functions-integrating-llm-technology-into-the-wolfram-language/), (2023), [Stephen Wolfram Writings](https://writings.stephenwolfram.com) .

[SW2] Stephen Wolfram, ["Introducing Chat Notebooks: Integrating LLMs into the Notebook Paradigm"](https://writings.stephenwolfram.com/2023/06/introducing-chat-notebooks-integrating-llms-into-the-notebook-paradigm/), (2023), [Stephen Wolfram Writings](https://writings.stephenwolfram.com) .

[SW3] Stephen Wolfram, ["Prompts for Work & Play: Launching the Wolfram Prompt Repository"](https://writings.stephenwolfram.com/2023/06/prompts-for-work-play-launching-the-wolfram-prompt-repository/),  (2023), [Stephen Wolfram Writings](https://writings.stephenwolfram.com) .

### Notebooks

[AAn1p6] Anton Antonov, ["Workflows with LLM functions (in Raku)"](https://community.wolfram.com/groups/-/m/t/2982320) , (2023), [community.wolfram.com](https://community.wolfram.com/) .

[AAn1wl] Anton Antonov, ["Workflows with LLM functions (in WL)"](https://community.wolfram.com/groups/-/m/t/2983602) , (2023), [community.wolfram.com](https://community.wolfram.com/) .

[AAn1py] Anton Antonov, ["Workflows with LLM functions (in Python)"]() , (2023), [community.wolfram.com](https://community.wolfram.com/) .

[AAn2] Anton Antonov, ["Raku, Python, and Wolfram Language over LLM functionalities"](https://community.wolfram.com/groups/-/m/t/3053519), (2023), [Wolfram Community](https://community.wolfram.com).

### Python packages

[AAp1py] Anton Antonov, [LLMFunctions Python package]() , (2023-2026), [PyPI.org/antononcube](https://pypi.org/user/antononcube/) .

[AAp2py] Anton Antonov, [LLMPrompts Python package]() , (2023-2026), [PyPI.org/antononcube](https://pypi.org/user/antononcube/) .

[AAp3py] Anton Antonov, [DataTypeSystem Python package](https://pypi.org/project/DataTypeSystem/) , (2023), [PyPI.org/antononcube](https://pypi.org/user/antononcube/) .

[AAp4py] Anton Antonov, [JupyterChatbook Python package](https://pypi.org/project/JupyterChatbook/) , (2023-2026), [PyPI.org/antononcube](https://pypi.org/user/antononcube/) .

### Raku packages

[AAp1p6] Anton Antonov, [LLM::Functions Raku package](https://raku.land/zef:antononcube/ML::FindTextualAnswer) , (2023-2026), [raku.land/antononcube](https://raku.land/zef:antononcube) .

[AAp2p6] Anton Antonov, [LLMPrompts Raku package](https://raku.land/zef:antononcube/LLM::Prompts) , (2023-2025), [raku.land/antononcube](https://raku.land/zef:antononcube) .

[AAp3p6] Anton Antonov, [Data::TypeSystem Raku package](https://raku.land/zef:antononcube/ML::TypeSystem) , (2023-2025), [raku.land/antononcube](https://raku.land/zef:antononcube) .

[AAp4p6] Anton Antonov, [Jupyter::Chatbook Raku package](https://raku.land/zef:antononcube/Jupyter::Chatbook) , (2023-2026), [raku.land/antononcube](https://raku.land/zef:antononcube) .

[AAp5p6] Anton Antonov, [ML::FindTextualAnswer Raku package](https://raku.land/zef:antononcube/ML::FindTextualAnswer) , (2023-2026), [raku.land/antononcube](https://raku.land/zef:antononcube) .

### Wolfram Language paclets

[WRIp1] Wolfram Research Inc., [LLMFunctions paclet](https://resources.wolframcloud.com/PacletRepository/resources/Wolfram/LLMFunctions/) , (2023) [Wolfram Paclet Repository](https://resources.wolframcloud.com/PacletRepository/) .

[WRIr1] Wolfram Research Inc., [Wolfram Prompt Repository](https://resources.wolframcloud.com/PromptRepository/) .

[AAp4wl] Anton Antonov, [NLPTemplateEngine paclet](https://resources.wolframcloud.com/PacletRepository/resources/AntonAntonov/NLPTemplateEngine/) , (2023) [Wolfram Paclet Repository](https://resources.wolframcloud.com/PacletRepository/) .

### Videos

[AAv1] Anton Antonov, ["Jupyter Chatbook LLM cells demo (Raku)"](https://www.youtube.com/watch?v=cICgnzYmQZg), (2023), [YouTube/@AAA4Prediction](https://www.youtube.com/@AAA4Prediction) .

[AAv2] Anton Antonov, ["Jupyter Chatbook multi-cell LLM chats demo (Raku)"](https://youtu.be/WN3N-K_Xzz8), (2023), [YouTube/@AAA4Prediction](https://www.youtube.com/@AAA4Prediction) .

[AAv3] Anton Antonov, ["Jupyter Chatbook LLM cells demo (Python)"](https://www.youtube.com/watch?v=WN3N-K_Xzz8) , (2023), [YouTube/@AAA4Prediction](https://www.youtube.com/@AAA4Prediction) .

[AAv4] Anton Antonov, ["Jupyter Chatbook multi cell LLM chats teaser (Python)"](https://www.youtube.com/watch?v=8pv0QRGc7Rw) , (2023), [YouTube/@AAA4Prediction](https://www.youtube.com/@AAA4Prediction) .

[AAv5] Anton Antonov, ["Simplified Machine Learning Workflows Overview (Raku-centric)](https://www.youtube.com/watch?v=p3iwPsc6e74) , (2022), [YouTube/@AAA4Prediction](https://www.youtube.com/@AAA4Prediction) .

[AAv6] Anton Antonov, ["Natural Language Processing Template Engine"](https://www.youtube.com/watch?v=IrIW9dB5sRM), (2022), WTC-2022,  [YouTube/@WolframResearch](https://www.youtube.com/@WolframResearch).